In [ ]:
%cd ../..
import os
import polars as pl
import numpy as np
import random

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from evaluation import *

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
embeddings_path = "/scratch/VM/radio-foundation/cache/embeddings/DeepRDT"

files = [x for x in os.listdir(embeddings_path) if x.endswith(".pth")]
id_to_path = {
    f.replace(".pth", "") : os.path.join(embeddings_path, f) for f in files
}

In [ ]:
data_path = "/mnt/typhon/data/AI/DeepRDT/DeepRDT_rectum/Deep_rectum_crop"

clinical_raw = pl.read_csv(os.path.join(data_path, "dfRectum_cropDetails_wClinic.csv"))
clinical_raw.head()

In [ ]:
label_name = "tumoral_regression_grade"
clinical_raw[label_name].value_counts()

In [ ]:
# sexe
df = clinical_raw.filter(pl.col(label_name).is_not_null())
labels = {
    str(key): int(value == "Home")
    for key, value in zip(df['MAPID'], df[label_name])
}
num_classes = 2

In [ ]:
# binary_response
df = clinical_raw
labels = {
    str(key): int(value)
    for key, value in zip(df['MAPID'], df[label_name])
}
num_classes = 2

In [ ]:
# tumoral_regression_grade
df = clinical_raw.filter(pl.col(label_name).is_not_null())
df = clinical_raw.filter(pl.col(label_name) < 5.0)
labels = {
    str(key): int(value > 2)
    for key, value in zip(df['MAPID'], df[label_name])
}
num_classes = 2

In [ ]:
patient_ids = list(labels.keys())
exist_patient_ids = [x for x in patient_ids if x in id_to_path.keys()]
exist_labels = [labels[pid] for pid in exist_patient_ids]
len(exist_labels)

In [ ]:
train_ids, val_ids = train_test_split(exist_patient_ids, test_size=0.2, random_state=5, stratify=exist_labels)

train_dataset = EmbeddingDataset(train_ids, id_to_path, labels, add_noise=True, p=1.0, sigma=0.1)
val_dataset = EmbeddingDataset(val_ids, id_to_path, labels)

In [ ]:
class_weights = get_class_weights(exist_labels)
class_weights

In [ ]:
EMBED_DIM = 768
num_epochs = 50
batch_size = 16
learning_rate = 0.0001

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=collate_classification,
    batch_size=batch_size,
)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_classification)

device = torch.device("cuda")
pooler = GatedPool(EMBED_DIM)
model = Classifier(pooler, embed_dim=EMBED_DIM, num_classes=num_classes).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

output = train_classifier(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device
)
model.load_state_dict(output["state_dict"])


In [ ]:
plot_train_curves(output["train_loss"], output["val_loss"], "Cross-Entropy Loss")

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, device)
all_predictions = torch.nn.functional.softmax(all_predictions, dim=1)
all_predictions = torch.argmax(all_predictions, dim=1)

plot_confusion_matrix(all_labels, all_predictions)